# LLM의 Context Window 이해하기 (OpenAI gpt-4.1-nano)

**한 줄 요약: LLM은 기억이 없습니다. 매번 대화 전체를 다시 읽습니다.**

우리가 챗봇과 대화하면 마치 LLM이 앞의 내용을 "기억"하는 것처럼 보입니다.
하지만 실제로 LLM은 이전 대화를 저장하지 않습니다 (stateless).
매 요청마다 **지금까지의 대화 전부**를 다시 통째로 보내주기 때문에 기억하는 것처럼 보일 뿐입니다.

그렇다면 무한정 보낼 수 있을까요? 아닙니다.
한 번에 넣을 수 있는 토큰의 최대 개수가 정해져 있고, 이것을 **Context Window** 라고 부릅니다.

| 모델 | Context Window (대략) |
|---|---|
| llama3.2 | 128,000 토큰 |
| gpt-4o | 128,000 토큰 |
| gpt-4.1-nano | 1,047,576 토큰 (약 100만) |
| Claude Sonnet | 200,000 토큰 |

이 노트북은 `1_7_2_context_ollama.ipynb` 와 같은 내용을 **로컬 Ollama 대신 OpenAI API(gpt-4.1-nano)** 로 실행합니다.
4단계로 직접 확인해 봅니다.

1. LLM은 기억이 없다 (stateless)
2. 기억 = 이전 대화를 다시 보내주는 것
3. 대화가 길어질수록 토큰이 쌓인다
4. Context Window가 꽉 차면 오래된 기억부터 잘린다

## 준비. OpenAI API 키 불러오기

day1과 동일하게 `.env` 파일에 `OPENAI_API_KEY` 가 들어 있어야 합니다.

```
OPENAI_API_KEY=sk-proj-...
```

> **주의:** OpenAI API는 유료입니다. gpt-4.1-nano는 가장 저렴한 모델 중 하나라 이 노트북 전체를 실행해도 비용은 1원 미만 수준입니다.

In [1]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key 확인: {openai_api_key[:8]}...")
else:
    print("OPENAI_API_KEY 가 없습니다. .env 파일을 확인하세요.")

# api_key, base_url 을 지정하지 않으면 환경변수 OPENAI_API_KEY 를 사용해 OpenAI 서버로 접속합니다.
openai = OpenAI()

MODEL = "gpt-4.1-nano"

OpenAI API Key 확인: sk-proj-...


> **참고:** `1_7_2_context_ollama.ipynb` 와 비교해 보면 바뀐 것은 **클라이언트 생성 부분과 모델 이름뿐**입니다.
> OpenAI SDK 사용법(`chat.completions.create`)은 Ollama와 완전히 같습니다.
>
> gpt-4.1-nano는 llama3.2(3B)보다 한국어를 훨씬 자연스럽게 답합니다.
> 하지만 이 노트북에서 볼 것은 여전히 **답변의 품질이 아니라 "기억하는가 / 못 하는가"** 입니다.
> 똑똑한 모델이라도 보내주지 않은 내용은 기억할 수 없습니다.

## 1단계. LLM은 기억이 없다 (stateless)

먼저 가장 단순한 형태로 질문해 봅니다.
`messages` 에 **지금 질문 하나만** 담아서 보냅니다.

In [2]:
def ask(question):
    """질문 하나만 보내는 가장 단순한 호출 (이전 대화 없음)"""
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
    )
    return response.choices[0].message.content

In [3]:
# 첫 번째 대화: 내 이름을 알려줍니다.
print(ask("내 이름은 상화야. 반가워!"))

반가워요, 상화님! 만나서 정말 반가워요. 어떻게 도와드릴까요?


In [4]:
# 두 번째 대화: 방금 알려준 이름을 물어봅니다.
print(ask("내 이름이 뭐라고 했지?"))

죄송하지만, 이전 대화 내용에 당신의 이름이 언급되지 않은 것 같아서요. 이름이 어떻게 되시나요?


### 결과

모델은 이름을 **모른다**고 답합니다.

두 번째 호출에서 우리가 보낸 것은 `"내 이름이 뭐라고 했지?"` 한 문장뿐이기 때문입니다.
첫 번째 대화는 어디에도 저장되어 있지 않습니다.

ChatGPT 웹사이트에서는 이름을 기억하는 것처럼 보이는데, 그것은 ChatGPT **앱**이 대화 기록을 대신 보내주기 때문입니다.
API로 모델을 직접 호출하면 그런 처리가 없습니다.

> LLM은 함수와 같습니다. 입력(messages)만 보고 출력을 만듭니다. 이전 호출은 전혀 알지 못합니다.

## 2단계. 기억 = 이전 대화를 다시 보내주는 것

이번에는 이전 대화를 `messages` 리스트에 직접 넣어서 보냅니다.
바뀐 것은 **모델이 아니라 우리가 보낸 입력**뿐입니다.

In [5]:
messages = [
    {"role": "user", "content": "내 이름은 상화야. 반가워!"},
    {"role": "assistant", "content": "반가워요, 상화님!"},
    {"role": "user", "content": "내 이름이 뭐라고 했지?"},
]

response = openai.chat.completions.create(model=MODEL, messages=messages)
print(response.choices[0].message.content)

상화라고 하셨죠!


### 결과

이번에는 이름을 정확히 맞춥니다.

챗봇의 "기억"은 사실 **대화 기록(messages)을 매번 다시 보내주는 것**입니다.
그리고 이렇게 매번 보내는 이 대화 기록 전체가 바로 **context** 입니다.

## 3단계. 대화가 길어질수록 토큰이 쌓인다

context는 공짜가 아닙니다. 대화가 한 턴 늘어날 때마다 보내야 하는 토큰도 늘어납니다.
실제로 몇 토큰을 보냈는지는 응답의 `usage` 에서 확인할 수 있습니다.

OpenAI API는 **토큰 수만큼 요금**을 받기 때문에, 보낸 토큰이 늘어난다는 것은 곧 **비용이 늘어난다**는 뜻입니다.

In [6]:
chat_history = []


def chat(question):
    """대화 기록을 계속 쌓아가며 보내는 함수"""
    chat_history.append({"role": "user", "content": question})

    response = openai.chat.completions.create(model=MODEL, messages=chat_history)
    answer = response.choices[0].message.content

    chat_history.append({"role": "assistant", "content": answer})

    # prompt_tokens = 이번에 우리가 '보낸' 토큰 수 = 지금 사용 중인 context 크기
    # completion_tokens = 모델이 '생성한' 답변의 토큰 수
    print(
        f"[보낸 토큰(context): {response.usage.prompt_tokens}, "
        f"받은 토큰(답변): {response.usage.completion_tokens}]"
    )
    return answer

In [7]:
print(chat("내 이름은 철수야."))

[보낸 토큰(context): 14, 받은 토큰(답변): 14]
안녕하세요, 철수님! 어떻게 도와드릴까요?


In [8]:
print(chat("나는 서울에 살아."))

[보낸 토큰(context): 41, 받은 토큰(답변): 25]
서울에 사시군요! 서울에 관한 이야기나 궁금한 점이 있으면 언제든 말씀해 주세요.


In [9]:
print(chat("내 취미는 등산이야."))

[보낸 토큰(context): 83, 받은 토큰(답변): 56]
좋은 취미를 가지고 계시네요, 철수님! 등산은 건강에도 좋고 자연을 만끽할 수 있어서 정말 멋진 취미인 것 같아요. 요즘 가고 싶은 산이나 추천하는 등산로가 있나요?


질문 자체는 비슷한 길이인데도 `보낸 토큰` 수가 계속 늘어납니다.
이전 대화(모델의 답변까지 포함)가 전부 다시 실려 가기 때문입니다.

**이 숫자가 모델의 Context Window(gpt-4.1-nano는 약 1,047,576)에 도달하면 더 이상 넣을 수 없습니다.**

> Context Window가 100만 토큰으로 크더라도, 그만큼 보내면 그만큼 요금이 나오고 응답도 느려집니다.
> 크다고 해서 무조건 다 넣는 것이 좋은 것은 아닙니다.

## 4단계. Context Window가 꽉 차면?

실제 모델의 100만 토큰을 채우려면 너무 오래 걸리고 비용도 들기 때문에,
**Context Window가 아주 작다고 가정**하고 흉내내 봅니다.

한계를 넘으면 보통 **오래된 대화부터 잘라냅니다** (sliding window).

토큰 수는 `tiktoken` 으로 계산합니다.
gpt-4.1 계열 모델은 `o200k_base` 토크나이저를 사용하므로 `encoding_for_model` 대신 이름으로 직접 지정합니다.

> 메시지마다 role 표시 등 몇 토큰의 추가 비용이 붙기 때문에, 여기서 계산한 값은 `usage.prompt_tokens` 보다 조금 작은 **근사치**입니다.

In [10]:
import tiktoken

encoding = tiktoken.get_encoding("o200k_base")  # gpt-4.1 계열이 사용하는 토크나이저


def count_tokens(messages):
    return sum(len(encoding.encode(m["content"])) for m in messages)

In [11]:
CONTEXT_WINDOW = 60  # 아주 작은 context window 라고 가정 (실제 gpt-4.1-nano는 1047576)


def trim_to_window(messages):
    """context window를 넘으면 가장 오래된 대화부터 버린다"""
    while len(messages) > 1 and count_tokens(messages) > CONTEXT_WINDOW:
        dropped = messages.pop(0)
        print(f"  ✂ 잘려나간 대화: [{dropped['role']}] {dropped['content'][:30]}...")
    return messages

In [12]:
small_history = []


def chat_with_small_window(question):
    small_history.append({"role": "user", "content": question})
    trim_to_window(small_history)

    print(f"  (현재 context: {count_tokens(small_history)} / {CONTEXT_WINDOW} 토큰)")

    response = openai.chat.completions.create(model=MODEL, messages=small_history)
    answer = response.choices[0].message.content

    small_history.append({"role": "assistant", "content": answer})
    return answer

In [13]:
print(chat_with_small_window("내 이름은 상화야. 꼭 기억해줘."))

  (현재 context: 12 / 60 토큰)
반갑습니다, 상화님! 앞으로 기억해둘게요. 무엇을 도와드릴까요?


In [14]:
print(chat_with_small_window("오늘 서울 날씨는 어때? 두 문장으로 답해줘."))

  (현재 context: 51 / 60 토큰)
죄송하지만, 저는 실시간 날씨 정보를 제공할 수 없습니다. 오늘 서울의 날씨를 확인하시려면 기상청이나 날씨 앱을 참고해 주세요.


In [15]:
print(chat_with_small_window("등산할 때 필요한 준비물을 알려줘. 두 문장으로 답해줘."))

  ✂ 잘려나간 대화: [user] 내 이름은 상화야. 꼭 기억해줘....
  ✂ 잘려나간 대화: [assistant] 반갑습니다, 상화님! 앞으로 기억해둘게요. 무엇을 도와...
  ✂ 잘려나간 대화: [user] 오늘 서울 날씨는 어때? 두 문장으로 답해줘....
  (현재 context: 55 / 60 토큰)
등산 시에는 편안한 등산화와 적당한 복장을 준비하는 것이 중요합니다. 또한, 물과 간단한 간식을 챙기고, 기상 변화에 대비할 수 있는 비상용품도 준비하세요.


In [16]:
# 이제 다시 이름을 물어봅니다.
print(chat_with_small_window("내 이름이 뭐라고 했지?"))

  ✂ 잘려나간 대화: [assistant] 죄송하지만, 저는 실시간 날씨 정보를 제공할 수 없습니...
  ✂ 잘려나간 대화: [user] 등산할 때 필요한 준비물을 알려줘. 두 문장으로 답해줘...
  (현재 context: 60 / 60 토큰)
죄송하지만, 지금까지 대화에서 사용자의 이름을 알려주지 않으셔서 이름을 알 수 없습니다. 혹시 이름을 알려주시면 기억해 두겠습니다!


### 결과

이름을 알려준 첫 대화가 **context window 밖으로 밀려나 잘렸기 때문에**
모델은 더 이상 이름을 알지 못합니다.

gpt-4.1-nano처럼 성능이 좋은 모델이라도 결과는 같습니다.
챗봇이 긴 대화 도중 앞부분을 "잊어버리는" 이유가 바로 이것입니다.
모델이 까먹은 것이 아니라, **애초에 보내주지 않은 것**입니다.

## 정리

| 개념 | 의미 |
|---|---|
| Stateless | LLM은 이전 호출을 기억하지 않는다 |
| Context | 이번 요청에 함께 보내는 대화 기록 전체 |
| Context Window | 한 번에 보낼 수 있는 최대 토큰 수 |
| 잘림(Truncation) | window를 넘으면 오래된 대화부터 버려진다 |

**Ollama(llama3.2) vs OpenAI(gpt-4.1-nano)**

| 항목 | llama3.2 (1_7_2_context_ollama.ipynb) | gpt-4.1-nano (이 노트북) |
|---|---|---|
| 실행 위치 | 내 컴퓨터 (로컬) | OpenAI 서버 (클라우드) |
| 비용 | 무료 | 토큰당 과금 |
| Context Window | 128,000 토큰 | 약 1,047,576 토큰 |
| 토크나이저 | 자체 토크나이저 | o200k_base |
| Stateless 여부 | 기억 없음 | 기억 없음 (동일) |

**기억하기**

- 챗봇의 기억력 = 개발자가 매번 다시 보내주는 대화 기록
- 모델이 바뀌어도, Context Window가 커져도 이 원리는 똑같다
- context가 길수록 비용과 응답 시간도 늘어난다 (유료 API에서는 특히 중요)
- 긴 대화에서는 앞부분을 요약해서 넣거나(summary), 필요한 부분만 찾아서 넣는(RAG) 방법을 쓴다